In [5]:
import math
import time
from typing import List, Dict, Any

In [6]:
# --- Step 1: Define Our Memory Registry with Ingestion Metadata ---
MOCK_MEMORY_STORE: List[Dict[str, Any]] = [
    {
        "id": "mem_01",
        "text": "User explicitly stated their name is Aryan Yadav and they are a student at IIT Ropar.",
        "importance_score": 9,
        "timestamp_unix": time.time() - (30 * 24 * 3600)  # Created 30 days ago (Old)
    },
    {
        "id": "mem_02",
        "text": "User is actively testing an XGBoost model regressor tracking solar material metrics.",
        "importance_score": 7,
        "timestamp_unix": time.time() - (600)  # Created 10 minutes ago (Highly Recent!)
    },
    {
        "id": "mem_03",
        "text": "User mentioned they prefer hot coffee over iced tea during a casual conversational break.",
        "importance_score": 1,
        "timestamp_unix": time.time() - (300)  # Created 5 minutes ago (Recent but trivial)
    }
]

In [7]:
# --- Step 2: The Multi-Factor Retrieval Core ---

class CognitiveRetrievalEngine:
    def __init__(self, recency_weight: float = 1.0, importance_weight: float = 1.0):
        self.w_recency = recency_weight
        self.w_importance = importance_weight
        print("[RETRIEVAL ENG] Scoring weights calibrated. Processing constraints online.")

    def _simulate_semantic_relevance(self, query: str, memory_text: str) -> float:
        """Simulates raw embedding cosine similarity matching score limits."""
        q_low, mem_low = query.lower(), memory_text.lower()
        
        # Simple simulation mapping semantic overlaps
        if "who am i" in q_low or "name" in q_low or "student" in q_low:
            if "aryan" in mem_low: return 0.95
        if "model" in q_low or "hackathon" in q_low or "solar" in q_low:
            if "xgboost" in mem_low: return 0.90
            
        return 0.15 # Baseline weak alignment floor

    def retrieve_top_memories(self, user_query: str, limit: int = 1) -> List[Dict[str, Any]]:
        print(f"\nEvaluating memory registry against active user intent: '{user_query}'...")
        scored_memories = []
        current_time = time.time()

        for memory in MOCK_MEMORY_STORE:
            # 1. Compute Semantic Core Correlation
            relevance = self._simulate_semantic_relevance(user_query, memory["text"])
            
            # 2. Compute Exponential Recency Decay Curve
            # Formula simulation: Score drops gracefully the older the timestamp delta is
            time_delta_days = (current_time - memory["timestamp_unix"]) / (24 * 3600)
            recency_score = math.exp(-0.1 * time_delta_days) # Decay rate multiplier
            
            # 3. Extract Normalized Importance Scaling
            importance_score = memory["importance_score"] / 10.0 # Map 1-10 down to 0.0-1.0
            
            # 4. Synthesize Combined Hybrid Vector Score
            # Total Score = Relevance + (Weighted Recency) + (Weighted Importance)
            total_retrieval_score = relevance + (self.w_recency * recency_score) + (self.w_importance * importance_score)
            
            scored_memories.append({
                "text": memory["text"],
                "metrics": {
                    "raw_relevance": round(relevance, 2),
                    "decayed_recency": round(recency_score, 2),
                    "importance": round(importance_score, 2),
                    "composite_rank": round(total_retrieval_score, 3)
                }
            })

        # Re-sort rank arrays descending based on structural composite scores
        scored_memories.sort(key=lambda x: x["metrics"]["composite_rank"], reverse=True)
        return scored_memories[:limit]

In [8]:
# --- Step 3: Test Scoring Decisions Across Different Intentions ---

engine = CognitiveRetrievalEngine(recency_weight=0.5, importance_weight=0.5)

# Test Run 1: Query searching for high-importance, older historical data
results_1 = engine.retrieve_top_memories(user_query="Identify my profile status and name details.", limit=1)
print(f"Top Extracted Memory Node:\n  » \"{results_1[0]['text']}\"\n  » Scores: {results_1[0]['metrics']}")

# Test Run 2: Query matching hyper-recent operational data points
results_2 = engine.retrieve_top_memories(user_query="What machine learning model did I just configure?", limit=1)
print(f"Top Extracted Memory Node:\n  » \"{results_2[0]['text']}\"\n  » Scores: {results_2[0]['metrics']}")

[RETRIEVAL ENG] Scoring weights calibrated. Processing constraints online.

Evaluating memory registry against active user intent: 'Identify my profile status and name details.'...
Top Extracted Memory Node:
  » "User explicitly stated their name is Aryan Yadav and they are a student at IIT Ropar."
  » Scores: {'raw_relevance': 0.95, 'decayed_recency': 0.05, 'importance': 0.9, 'composite_rank': 1.425}

Evaluating memory registry against active user intent: 'What machine learning model did I just configure?'...
Top Extracted Memory Node:
  » "User is actively testing an XGBoost model regressor tracking solar material metrics."
  » Scores: {'raw_relevance': 0.9, 'decayed_recency': 1.0, 'importance': 0.7, 'composite_rank': 1.75}
